Qdrant DB 참고 문서 : https://wikidocs.net/377800

# 1. Qdrant 및 OpenAI 연결

In [2]:
import os
from dotenv import load_dotenv
from common.ai_model import get_llm_model, get_embedding_model
from common.qdrant import get_qdrant_client

load_dotenv(override=True)  # .env의 API Key, URL 등 환경변수 로드

llm = get_llm_model()  # LLM(ChatOpenAI) 클라이언트 생성

qdrant_client = get_qdrant_client()  # Qdrant 벡터DB 클라이언트 생성

print(qdrant_client.get_collections())  # 현재 생성된 컬렉션 목록 확인 (초기에는 비어있음)

collections=[CollectionDescription(name='star_charts'), CollectionDescription(name='terraforming')]


# 2. 법률문서 더미데이터 준비

In [3]:
# RAG 테스트용 법률문서 더미데이터 (실제 서비스에서는 DB/파일 등에서 로드)
documents = [
    {
        "id": 1,
        "article": "제1조",
        "title": "목적",
        "content": "이 법은 인공지능의 건전한 발전과 신뢰 기반 조성에 필요한 기본적인 사항을 규정함으로써 국민의 권익과 존엄성을 보호하고 국민의 삶의 질 향상과 국가경쟁력을 강화하는 데 이바지함을 목적으로 한다."
    },
    {
        "id": 2,
        "article": "제2조",
        "title": "정의",
        "content": '''이 법에서 사용하는 용어의 뜻은 다음과 같다. <개정 2026. 1. 20.>
1. 인공지능"이란 학습, 추론, 지각, 판단, 언어의 이해 등 인간이 가진 지적 능력을 전자적 방법으로 구현한 것을 말한다.
2. "인공지능시스템"이란 다양한 수준의 자율성과 적응성을 가지고 주어진 목표를 위하여 실제 및 가상환경에 영향을 미치는 예측, 추천, 결정 등의 결과물을 추론하는 인공지능 기반 시스템을 말한다.
3. "인공지능기술"이란 인공지능을 구현하기 위하여 필요한 하드웨어ㆍ소프트웨어 기술 또는 그 활용 기술을 말한다.
4. "고영향 인공지능"이란 사람의 생명, 신체의 안전 및 기본권에 중대한 영향을 미치거나 위험을 초래할 우려가 있는 인공지능시스템으로서 다음 각 목의 어느 하나의 영역에서 활용되는 것을 말한다.
가. 「에너지법」 제2조제1호에 따른 에너지의 공급
나. 「먹는물관리법」 제3조제1호에 따른 먹는물의 생산 공정
다. 「보건의료기본법」 제3조제1호에 따른 보건의료의 제공 및 이용체계의 구축ㆍ운영
라. 「의료기기법」 제2조제1항에 따른 의료기기 및 「디지털의료제품법」 제2조제2호에 따른 디지털의료기기의 개발 및 이용
마. 「원자력시설 등의 방호 및 방사능 방재 대책법」 제2조제1항제1호에 따른 핵물질과 같은 항 제2호에 따른 원자력시설의 안전한 관리 및 운영
바. 범죄 수사나 체포 업무를 위한 생체인식정보(얼굴ㆍ지문ㆍ홍채 및 손바닥 정맥 등 개인을 식별할 수 있는 신체적ㆍ생리적ㆍ행동적 특징에 관한 개인정보를 말한다)의 분석ㆍ활용
사. 채용, 대출 심사 등 개인의 권리ㆍ의무 관계에 중대한 영향을 미치는 판단 또는 평가
아. 「교통안전법」 제2조제1호부터 제3호까지에 따른 교통수단, 교통시설, 교통체계의 주요한 작동 및 운영
자. 공공서비스 제공에 필요한 자격 확인 및 결정 또는 비용징수 등 국민에게 영향을 미치는 국가, 지방자치단체, 「공공기관의 운영에 관한 법률」 제4조에 따른 공공기관 등(이하 "국가기관등"이라 한다)의 의사결정
차. 「교육기본법」 제9조제1항에 따른 유아교육ㆍ초등교육 및 중등교육에서의 학생 평가
카. 그 밖에 사람의 생명ㆍ신체의 안전 및 기본권 보호에 중대한 영향을 미치는 영역으로서 대통령령으로 정하는 영역
5. "생성형 인공지능"이란 입력한 데이터(「데이터 산업진흥 및 이용촉진에 관한 기본법」 제2조제1호에 따른 데이터를 말한다. 이하 같다)의 구조와 특성을 모방하여 글, 소리, 그림, 영상, 그 밖의 다양한 결과물을 생성하는 인공지능시스템을 말한다.
6. "인공지능산업"이란 인공지능 또는 인공지능기술을 활용한 제품(이하 "인공지능제품"이라 한다)을 개발ㆍ제조ㆍ생산 또는 유통하거나 이와 관련한 서비스(이하 "인공지능서비스"라 한다)를 제공하는 산업을 말한다.
7. "인공지능사업자"란 인공지능산업과 관련된 사업을 하는 자로서 다음 각 목의 어느 하나에 해당하는 법인, 단체, 개인 및 국가기관등을 말한다.
가. 인공지능개발사업자: 인공지능을 개발하여 제공하는 자
나. 인공지능이용사업자: 가목의 사업자가 제공한 인공지능을 이용하여 인공지능제품 또는 인공지능서비스를 제공하는 자
8. "이용자"란 인공지능제품 또는 인공지능서비스를 제공받는 자를 말한다.
9. "영향받는 자"란 인공지능제품 또는 인공지능서비스에 의하여 자신의 생명, 신체의 안전 및 기본권에 중대한 영향을 받는 자를 말한다.
10. "인공지능사회"란 인공지능을 통하여 산업ㆍ경제, 사회ㆍ문화, 행정 등 모든 분야에서 가치를 창출하고 발전을 이끌어가는 사회를 말한다.
11. "인공지능윤리"란 인간의 존엄성에 대한 존중을 기초로 하여, 국민의 권익과 생명ㆍ재산을 보호할 수 있는 안전하고 신뢰할 수 있는 인공지능사회를 구현하기 위하여 인공지능의 개발, 제공 및 이용 등 모든 영역에서 사회구성원이 지켜야 할 윤리적 기준을 말한다.
12. "학습용데이터"란 인공지능의 개발ㆍ활용 등에 사용되는 데이터를 말한다.'''
    },
    {
        "id": 3,
        "article": "제3조",
        "title": "기본원칙 및 국가 등의 책무",
        "content": '''① 인공지능기술과 인공지능산업은 안전성과 신뢰성을 제고하여 국민의 삶의 질을 향상시키는 방향으로 발전되어야 한다.
② 영향받는 자는 인공지능의 최종결과 도출에 활용된 주요 기준 및 원리 등에 대하여 기술적ㆍ합리적으로 가능한 범위에서 명확하고 의미 있는 설명을 제공받을 수 있어야 한다.
③ 국가 및 지방자치단체는 인공지능사업자의 창의정신을 존중하고, 안전한 인공지능 이용환경의 조성을 위하여 노력하여야 한다.
④ 국가 및 지방자치단체는 인공지능이 가져오는 사회ㆍ경제ㆍ문화와 국민의 일상생활 등 모든 영역에서의 변화에 대응하여 모든 국민이 안정적으로 적응할 수 있도록 시책을 강구하여야 한다.'''
    }
]

# 3. OpenAI Embedding 생성

In [4]:
embedding_client = get_embedding_model()  # 문서/질의를 벡터로 변환할 Embedding 모델 생성

# 4. 문서 임베딩 생성

In [7]:
COLLECTION_NAME = "law_articles"  # Qdrant 네임스페이스 역할을 하는 컬렉션 이름

# 컬렉션에 documents와 동일한 개수의 포인트가 이미 있으면 재임베딩하지 않음 (임베딩 API 호출 비용 절감)
needs_indexing = not (
    qdrant_client.collection_exists(COLLECTION_NAME)
    and qdrant_client.count(COLLECTION_NAME).count == len(documents)
)

if needs_indexing:
    # 2번에서 준비한 법률문서를 임베딩 벡터로 변환
    texts = [doc["content"] for doc in documents]
    vectors = await embedding_client.aembed_documents(texts)  # 문서별 임베딩 벡터 리스트
    print(f"문서 개수: {len(vectors)}")
    print(f"임베딩 차원: {len(vectors[0])}")
else:
    print(f"'{COLLECTION_NAME}'에 이미 {len(documents)}개 포인트가 저장되어 있어 임베딩을 건너뜁니다.")

AuthenticationError: Error code: 401 - {'error': {'message': 'Incorrect API key provided: <your-op*********key>. You can find your API key at https://platform.openai.com/account/api-keys.', 'type': 'invalid_request_error', 'param': None, 'code': 'invalid_api_key'}}

# 5. Qdrant 컬렉션 생성 및 저장

법률문서는 `law_articles`라는 네임스페이스(컬렉션)에 저장한다.

- Qdrant 유사도 이해  

| Metric | 핵심 개념 | 값의 의미 | 주요 용도 | Qdrant 설정 |
|---|---|---|---|---|
| **Cosine** | 벡터 방향의 유사성 | **값이 클수록 유사함** | NLP, Semantic Search, RAG | `Distance.COSINE` |
| **Dot Product** | 방향 + 크기 | **값이 클수록 유사함** | 추천 시스템, Ranking | `Distance.DOT` |
| **Euclidean** | 두 점 사이의 직선거리 | **값이 작을수록 유사함** | 이미지, 클러스터링, 수치 데이터 | `Distance.EUCLID` |
| **Manhattan** | 좌표 차이의 절댓값 합 | **값이 작을수록 유사함** | 수치 데이터, 일부 Sparse 데이터 | `Distance.MANHATTAN` |

In [70]:
from qdrant_client.models import Distance, PointStruct, VectorParams

if needs_indexing:
    VECTOR_SIZE = len(vectors[0])  # 저장할 벡터의 차원 수 (임베딩 모델 출력 차원과 일치해야 함)

In [71]:
# 컬렉션이 없을 때만 생성 (재실행 시 중복 생성 방지)
if needs_indexing and not qdrant_client.collection_exists(COLLECTION_NAME):
    qdrant_client.create_collection(
        collection_name=COLLECTION_NAME,
        vectors_config=VectorParams(
            size=VECTOR_SIZE, 
            distance=Distance.COSINE),  # 코사인 유사도 기반 검색(값이 클수록 유사함)
    )

In [72]:
if needs_indexing:
    # 문서별 벡터 + 메타데이터(payload)를 Qdrant 포인트로 구성
    points = [
        PointStruct(
            id=doc["id"],  # 포인트 고유 id (동일 id로 upsert하면 기존 포인트를 덮어씀)
            vector=vector,  # 해당 문서의 임베딩 벡터 (유사도 검색에 사용)
            payload={  # 벡터와 함께 저장되는 메타데이터 (검색 결과에서 원문 조회용)
                "article": doc["article"],
                "title": doc["title"],
                "content": doc["content"],
            },
        )
        for doc, vector in zip(documents, vectors)  # 문서와 임베딩 벡터를 같은 순서로 짝지어 순회
    ]

    qdrant_client.upsert(collection_name=COLLECTION_NAME, points=points)  # 컬렉션에 포인트 저장(동일 id면 덮어씀)

print(qdrant_client.get_collections())  # law_articles 컬렉션이 생성되었는지 확인

collections=[CollectionDescription(name='law_articles')]


# 6. 사용자 질의 검색 (Retrieval)

In [73]:
query = "인공지능 법의 목적이 뭐야?"  # 사용자 질의

query_vector = embedding_client.embed_query(query)  # 질의를 문서와 동일한 벡터 공간으로 임베딩

# 질의 벡터와 가장 유사한 문서 top-2 검색
search_result = qdrant_client.query_points(
    collection_name=COLLECTION_NAME,
    query=query_vector,
    limit=2,
)

for point in search_result.points:
    print(f"[{point.payload['article']} {point.payload['title']}] score={point.score:.4f}")
    print(point.payload["content"])
    print()

[제1조 목적] score=0.6677
이 법은 인공지능의 건전한 발전과 신뢰 기반 조성에 필요한 기본적인 사항을 규정함으로써 국민의 권익과 존엄성을 보호하고 국민의 삶의 질 향상과 국가경쟁력을 강화하는 데 이바지함을 목적으로 한다.

[제2조 정의] score=0.6084
이 법에서 사용하는 용어의 뜻은 다음과 같다. <개정 2026. 1. 20.>
1. 인공지능"이란 학습, 추론, 지각, 판단, 언어의 이해 등 인간이 가진 지적 능력을 전자적 방법으로 구현한 것을 말한다.
2. "인공지능시스템"이란 다양한 수준의 자율성과 적응성을 가지고 주어진 목표를 위하여 실제 및 가상환경에 영향을 미치는 예측, 추천, 결정 등의 결과물을 추론하는 인공지능 기반 시스템을 말한다.
3. "인공지능기술"이란 인공지능을 구현하기 위하여 필요한 하드웨어ㆍ소프트웨어 기술 또는 그 활용 기술을 말한다.
4. "고영향 인공지능"이란 사람의 생명, 신체의 안전 및 기본권에 중대한 영향을 미치거나 위험을 초래할 우려가 있는 인공지능시스템으로서 다음 각 목의 어느 하나의 영역에서 활용되는 것을 말한다.
가. 「에너지법」 제2조제1호에 따른 에너지의 공급
나. 「먹는물관리법」 제3조제1호에 따른 먹는물의 생산 공정
다. 「보건의료기본법」 제3조제1호에 따른 보건의료의 제공 및 이용체계의 구축ㆍ운영
라. 「의료기기법」 제2조제1항에 따른 의료기기 및 「디지털의료제품법」 제2조제2호에 따른 디지털의료기기의 개발 및 이용
마. 「원자력시설 등의 방호 및 방사능 방재 대책법」 제2조제1항제1호에 따른 핵물질과 같은 항 제2호에 따른 원자력시설의 안전한 관리 및 운영
바. 범죄 수사나 체포 업무를 위한 생체인식정보(얼굴ㆍ지문ㆍ홍채 및 손바닥 정맥 등 개인을 식별할 수 있는 신체적ㆍ생리적ㆍ행동적 특징에 관한 개인정보를 말한다)의 분석ㆍ활용
사. 채용, 대출 심사 등 개인의 권리ㆍ의무 관계에 중대한 영향을 미치는 판단 또는 평가
아. 「교통안전법」 제2조제1호부터 제3호까지에 따른 교통수단, 교통시설,

# 7. 컨텍스트 증강 생성 (RAG Generation)

In [74]:
# 검색된 문서들을 하나의 컨텍스트 문자열로 결합
context = "\n".join(
    f"{point.payload['article']} ({point.payload['title']}): {point.payload['content']}"
    for point in search_result.points
)

In [75]:
from pydantic import BaseModel, Field
from langchain_core.prompts import ChatPromptTemplate

# 평가하기 좋은 구조: 자유 텍스트가 아니라 "필드" 단위로 받아서 채점(정답 비교, 인용 검증 등)을 쉽게 함
class RagAnswer(BaseModel):
    answer: str = Field(description="사용자 질문에 대한 답변")
    cited_articles: list[str] = Field(description="답변의 근거가 된 조문 번호 목록 (예: ['제1조'])")
    is_answerable: bool = Field(description="주어진 컨텍스트만으로 답변 가능했는지 여부")

In [76]:
# prompt 객체화: context/question을 변수로 받는 재사용 가능한 템플릿 (하드코딩된 f-string 대신)
prompt = ChatPromptTemplate.from_messages([
    (
        "system",
        """당신은 주어진 법률 조문만 근거로 답변하는 법률 QA 어시스턴트다. 
        컨텍스트에 없는 내용은 추측하거나 지어내지 않는다. 
        컨텍스트만으로 답변할 수 없으면 is_answerable을 false로 설정한다."""
    ),
    (
      "human", 
      "context : {context}, question: {question}"
    ),
])

In [77]:
# outputparser: Pydantic 스키마로 구조화된 출력을 강제 (MonoRouter는 function_calling 방식에서만 정상 동작)
structured_llm = llm.with_structured_output(RagAnswer, method="function_calling")

In [78]:
# LCEL 체인 구성: prompt -> structured_llm
chain = prompt | structured_llm

# 이미 result가 있으면 LLM을 다시 호출하지 않음 (호출마다 비용이 발생하므로 재실행 방지)
if "result" not in globals():
    result = chain.invoke({"context": context, "question": query})  # RagAnswer 객체로 결과 수신
else:
    print("이미 생성된 result가 있어 LLM 호출을 건너뜁니다.")

이미 생성된 result가 있어 LLM 호출을 건너뜁니다.


# LLM에 갔다온 결과는 재실행하지 않도록 한다.
# 한번 갔다가 올때마다, 비용이 발생한다.

In [79]:
print(f"response result: {result}")

response result: answer='인공지능 법의 목적은 인공지능의 건전한 발전과 신뢰 기반 조성을 목적으로 합니다.' cited_articles=['제1조'] is_answerable=True


In [80]:
print("answer:", result.answer)
print("cited_articles:", result.cited_articles)
print("is_answerable:", result.is_answerable)

answer: 인공지능 법의 목적은 인공지능의 건전한 발전과 신뢰 기반 조성을 목적으로 합니다.
cited_articles: ['제1조']
is_answerable: True
